In [1]:
import numpy as np
import fmdj
from functools import cmp_to_key
import jax
import jax.numpy as jnp
import custom_jax as cj

Running cmake --build & --install in /home/jens/repos/custom-jax/build
Running cmake --build & --install in /home/jens/repos/custom-jax/build


# Argsort

In [13]:
i1 = jax.random.randint(jax.random.PRNGKey(0), (64,), 0, 2**20).block_until_ready()

isort1a = cj.tree.argsort(i1)
isort1b = jnp.argsort(i1)
print(jnp.all(isort1a == isort1b))

True


# Lexsort

In [2]:
i3 = jax.random.randint(jax.random.PRNGKey(0), (64,3), 0, 2**20).block_until_ready()
ilexa = jnp.lexsort(i3.T[::-1])
ilexb = cj.tree.i3zsort(i3)
print(ilexa)
print(ilexb)
print(jnp.all(ilexa == ilexb))

[43 53 52 48  1 60 14  3 21 41 18 32 26 50 35 10  9 11 59 30 57 12  6 28
 20 27 25 24 62 56 51 19 23 17 39 54 45 58 61  5 34 37 49 63 36  4 46 47
 55  7 31 40  2 16 13  0 42 33  8 44 29 22 15 38]
[43 53 52 48  1 60 14  3 21 41 18 32 26 50 35 10  9 11 59 30 57 12  6 28
 20 27 25 24 62 56 51 19 23 17 39 54 45 58 61  5 34 37 49 63 36  4 46 47
 55  7 31 40  2 16 13  0 42 33  8 44 29 22 15 38]
True


# I3 Zsort

In [2]:
def less_msb(x:int, y:int):
    return (x < y) and (x < (x ^ y))

def compare_zorder_int(a, b):
    assert np.all(a >= 0) and np.all(b >= 0), "Inputs must be non-negative"
    assert a.dtype == np.int32
    aint, bint = a.view(np.int32), b.view(np.int32)

    msd = 0
    for d in range(1, len(a)):
        if less_msb(aint[msd] ^ bint[msd], aint[d] ^ bint[d]):
            msd = d

    return -1*(a[msd] < b[msd])  + 1 * (a[msd] > b[msd])

In [3]:
x3 = np.float32(np.random.uniform(-0.5,0.5, (64,3)))
i3, valid = fmdj.octree.pos_to_icoord(x3)
# m3 = fmdj.octree.morton_3int32_to_3int32(i3)
i3zsort_jax = fmdj.octree.organize_particles(x3, return_sorted=False)[1]
i3zsort_py = np.array(sorted(range(len(x3)), key=cmp_to_key(lambda i, j: compare_zorder_int(i3[i], i3[j]))))
i3zsort_ffi = cj.tree.i3zsort(i3)

print(jnp.all(i3zsort_jax == i3zsort_py))
print(jnp.all(i3zsort_ffi == i3zsort_py))

True
True


# F3 Zsort

In [6]:
def xor_msb(a:np.float32, b:np.float32):
    """returns the most significant bit that differs between binary. repr. of floats"""
    if np.sign(a) != np.sign(b):
        return 128
    
    aint = abs(a).view(np.uint32)
    bint = abs(b).view(np.uint32)
    
    exp_a, exp_b = (aint >> 23)-127, (bint >> 23)-127

    if exp_a == exp_b: # If both floats have the same exponent, we need to compare mantissas
        most_sign_bit = int(aint ^ bint).bit_length()
        return exp_a + (most_sign_bit - 24)
    else: # Otherwise the most significant bit corresponds to the larger exponent
        return np.maximum(exp_a, exp_b)

def compare_zorder_float(a, b):
    assert a.dtype == np.float32

    msd = 0
    for d in range(1,3):
        if xor_msb(a[msd], b[msd]) < xor_msb(a[d], b[d]):
            msd = d

    return -1*(a[msd] < b[msd])  + 1 * (a[msd] > b[msd])

x3s = x3 - 0.5
f3zsort_py = np.array(sorted(range(len(x3)), key=cmp_to_key(lambda i, j: compare_zorder_float(np.float32(x3s[i]), np.float32(x3s[j])))))
f3zsort_ffi = cj.tree.f3zsort(jnp.float32(x3s))

print(jnp.all(f3zsort_py == i3zsort_jax))
print(jnp.all(f3zsort_ffi == i3zsort_jax))

print(i3zsort_jax)
print(f3zsort_py)
print(f3zsort_ffi)
# print(f3zsort_py)

True
True
[ 8  3 14 21 27 51 63 23  5 35 38  4 18  1 46 30 55 29 57 10 44 15 16 43
 37 26 61 25 13 59 47 41 40 28  6  2 36 42 58 60 48 49 54  9 32 45 62 12
  0 20 11 53 39 19 24 17  7 50 33 52 31 22 56 34]
[ 8  3 14 21 27 51 63 23  5 35 38  4 18  1 46 30 55 29 57 10 44 15 16 43
 37 26 61 25 13 59 47 41 40 28  6  2 36 42 58 60 48 49 54  9 32 45 62 12
  0 20 11 53 39 19 24 17  7 50 33 52 31 22 56 34]
[ 8  3 14 21 27 51 63 23  5 35 38  4 18  1 46 30 55 29 57 10 44 15 16 43
 37 26 61 25 13 59 47 41 40 28  6  2 36 42 58 60 48 49 54  9 32 45 62 12
  0 20 11 53 39 19 24 17  7 50 33 52 31 22 56 34]


In [31]:
(-3).bit_length()

2

In [26]:
# a = np.array((-1e20,), dtype=np.float32)
a = np.float32(10**np.random.uniform(-10., 10., 50))
b = np.float32(10**np.random.uniform(-10., 10., 50))

print(np.all((a < b) == (a.view(np.int32) < b.view(np.int32))))

True


In [29]:
print(a)
print((a.view(np.int32) >> 23) - 127)

[8.5942030e+00 4.9783921e-07 2.4642941e+09 3.1816797e+05 6.1656493e+08
 7.8803776e+07 9.0781967e+01 2.7383232e+08 5.8864038e-09 8.3110757e-07
 8.0268614e-10 3.2107509e-04 5.9367092e-03 2.6728872e-05 3.7562128e+06
 1.3647922e-03 4.1561401e-05 7.1436744e+00 6.5839195e-08 1.1064699e+06
 3.0975111e-09 5.4668698e+09 2.7854500e+05 9.4257001e-07 1.2895623e-10
 2.0381145e+06 1.3713455e+04 3.8031492e+04 2.6632194e+05 3.0261680e-09
 1.4767759e-03 2.0767356e-08 1.8283886e+07 2.9239001e+02 4.1491631e-04
 1.8670976e-09 1.6582364e-04 3.1890874e-04 3.9095211e+04 5.6383160e+02
 5.5495128e+07 2.7816245e-01 2.4653859e-08 1.8403301e+04 1.6432398e+05
 1.6809080e+01 2.6262956e+05 7.5147051e-01 2.8487647e+00 3.5548426e-02]
[  3 -21  31  18  29  26   6  28 -28 -21 -31 -12  -8 -16  21 -10 -15   2
 -24  20 -29  32  18 -21 -33  20  13  15  18 -29 -10 -26  24   8 -12 -29
 -13 -12  15   9  25  -2 -26  14  17   4  18  -1   1  -5]


In [25]:
xtest = np.array((-1., -1e-3, 1e-3, 0.01, 1., 1e3), dtype=np.float32)
print(sign_exp_mantissa(xtest))

(array([ True,  True, False, False, False, False]), array([  0, -10, -10,  -7,   0,   9], dtype=int32), array([      0,  201327,  201327, 2348810,       0, 7995392], dtype=int32))


In [15]:
print(x3[0,0], get_actual_exponent(x3[0,0]))
print(x3[0,1], get_actual_exponent(x3[0,1]))

-0.12545988 -3
0.45071432 -2
